# Capstone Two — Preprocessing and Training Data Development

## Predictive Healthcare Operations & Capacity Analytics

This notebook prepares the California hospital quarterly dataset for predictive modeling.

The prediction objective is to **predict next-quarter total hospital discharges (`DIS_TOT`) using information from the previous quarter**.

The notebook follows the Springboard Capstone Two Step 4 requirements:

1. Prepare a supervised dataset by matching each hospital's previous-quarter information with its next-quarter discharge total.
2. Review candidate features and missing values.
3. Split the data chronologically into training, validation, and testing datasets.
4. Impute missing values.
5. Encode categorical variables using indicator (dummy) variables.
6. Standardize numerical features.
7. Save the prepared datasets and preprocessing pipeline.

**Important:** All preprocessing steps are fitted using the training data only. The fitted transformations are then applied to validation and test data to prevent data leakage.

## 1. Import Libraries

Import the libraries needed for data preparation, chronological splitting, preprocessing, and saving the outputs.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load the Processed Dataset

Load the cleaned HCAI hospital dataset created during the Data Wrangling stage.

In [2]:
# Path to the processed HCAI hospital dataset.
DATA_PATH = Path("../data/processed/hcai_hospital_data_clean.csv")

# Load the dataset.
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (9179, 159)


## 3. Review the Dataset

Confirm that the hospital identifier, reporting period, target variable, and candidate features are available.

In [3]:
# Display the first five rows.
display(df.head())

# Review the required columns.
required_columns = [
    "FAC_NO",
    "YEAR_QTR",
    "DIS_TOT",
    "LIC_BEDS",
    "AVL_BEDS",
    "STF_BEDS",
    "DAY_TOT",
    "VIS_TOT",
    "TYPE_CNTRL",
    "TYPE_HOSP",
    "COUNTY_NAME",
    "HSA"
]

missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("All required columns are available.")
print("Number of facilities:", df["FAC_NO"].nunique())
print("Reporting periods:", sorted(df["YEAR_QTR"].dropna().unique()))

,FAC_NO,FAC_NAME,YEAR_QTR,BEG_DATE,END_DATE,OP_STATUS,COUNTY_NAME,HSA,HFPA,TYPE_CNTRL,TYPE_HOSP,TEACH_RURL,PHONE,ADDRESS,CITY,ZIP_CODE,CEO,LIC_BEDS,AVL_BEDS,STF_BEDS,DIS_MCAR,DIS_MCAR_MC,DIS_MCAL,DIS_MCAL_MC,DIS_CNTY,DIS_CNTY_MC,DIS_THRD,DIS_THRD_MC,DIS_INDGNT,DIS_OTH,DIS_TOT,DIS_LTC,DAY_MCAR,DAY_MCAR_MC,DAY_MCAL,DAY_MCAL_MC,DAY_CNTY,DAY_CNTY_MC,DAY_THRD,DAY_THRD_MC,DAY_INDGNT,DAY_OTH,DAY_TOT,DAY_LTC,VIS_MCAR,VIS_MCAR_MC,VIS_MCAL,VIS_MCAL_MC,VIS_CNTY,VIS_CNTY_MC,VIS_THRD,VIS_THRD_MC,VIS_INDGNT,VIS_OTH,VIS_TOT,GRIP_MCAR,GRIP_MCAR_MC,GRIP_MCAL,GRIP_MCAL_MC,GRIP_CNTY,GRIP_CNTY_MC,GRIP_THRD,GRIP_THRD_MC,GRIP_INDGNT,GRIP_OTH,GRIP_TOT,GROP_MCAR,GROP_MCAR_MC,GROP_MCAL,GROP_MCAL_MC,GROP_CNTY,GROP_CNTY_MC,GROP_THRD,GROP_THRD_MC,GROP_INDGNT,GROP_OTH,GROP_TOT,BAD_DEBT,CADJ_MCAR,CADJ_MCAR_MC,CADJ_MCAL,CADJ_MCAL_MC,DISP_855,CADJ_CNTY,CADJ_CNTY_MC,CADJ_THRD,CADJ_THRD_MC,CHAR_HB,CHAR_OTH,SUB_INDGNT,TCH_ALLOW,TCH_SUPP,DED_OTH,DED_TOT,CAP_MCAR,CAP_MCAL,CAP_CNTY,CAP_THRD,CAP_TOT,NET_MCAR,NET_MCAR_MC,NET_MCAL,NET_MCAL_MC,NET_CNTY,NET_CNTY_MC,NET_THRD,NET_THRD_MC,NET_INDGNT,NET_OTH,NET_TOT,OTH_OP_REV,TOT_OP_EXP,PHY_COMP,NONOP_REV,DIS_PIPS,DAY_PIPS,EXP_PIPS,EXP_POPS,CAP_EXP,FIX_ASSETS,DISP_TRNFR,DIS_TOT_CC,PAT_DAY_TOT_CC,TOT_OUT_VIS_CC,GROS_INPAT_REV_CC,GROS_OUTPAT_REV_CC,CONTR_ADJ_CC,OTHR_DEDUCT_CC,CAP_PREM_REV_CC,NET_PAT_REV_CC,QA_FEES,QA_SUPPL_PAY,MNGD_CARE_QA_PAY,DEP_AMORT_EXP_OPS,CASH,MARKETABLE_SEC,OTH_CURR_ASSETS,TOT_CURR_ASSETS,LIMITED_USE_CASH,LIMITED_USE_INV,LIMITED_USE_OTHER_ASSETS,TOT_LIMITED_USE_ASSETS,TOT_PPE,ACCU_DEP_AMORT,NET_TOT_PPE,CONST_IN_PROGRESS,TOT_INV_AND_OTH_ASSETS,TOT_INTANGIBLE_ASSETS,TOT_ASSETS,CURR_LIABILITIES,CURR_MAT_LT_DEBT,TOT_CURR_LIABILITIES,TOT_DEF_CREDITS,TOT_LT_DEBT,CURR_MATURITIES,NET_TOT_LT_DEBT,TOT_LIABILITIES,TOT_EQUITY,TOT_LIABILITY_AND EQUITY
0,106580996,ADVENTIST HEALTH AND RIDEOUT,20211,2021-01-01,2021-03-31,Open,Yuba,02 - Golden Empire,227,Non Profit Corp.,Comparable,NaN,530-749-4663,726 FOURTH ST,MARYSVILLE,95901,RICK RAWSON,221,221,185,811,186,164,1185,0,0,0,489,0,33,2868,0,3646,880,1271,8148,0,0,0,2154,0,157,16256,0,11313,1645,1661,15191,0,0,2734,7911,0,3276,43731,153232128,21311728,20282308,68693370,0,0,0,53699088,0,2105708,319324330,52970567,5726303,6806305,40600360,0,0,2058123,33444843,0,3403582,145010083,2666388,161249005,14670643,19080334,101116562,0,0,0,678333,44184127,0,2842566,0,0,0,3958689,350446647,0,0,0,0,0,44296689,12264567,7587394,8160436,0,0,501393,40425126,0,652161,113887766,2051738,128266608,0,75820,0,0,0,0,1686777,"318,254,966.000",0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,106150788,ADVENTIST HEALTH BAKERSFIELD,20211,2021-01-01,2021-03-31,Open,Kern,09 - Central,617,Non Profit Corp.,Comparable,NaN,661-395-3000,2615 CHESTER AVENUE,BAKERSFIELD,93301,SHARLET BRIGGS,254,254,197,696,851,173,772,0,0,251,964,0,14,3721,0,3751,3778,1205,3004,0,0,1480,4031,0,112,17361,0,12764,3932,1315,11424,0,0,5558,15592,0,1049,51634,78830087,85319575,18988853,66044326,0,0,29055733,89898131,0,1935857,370072562,46007280,33222211,4322872,53067097,0,0,14299424,67270111,0,4307177,222496172,2463652,104649244,100017020,17212027,95386197,0,0,0,29719714,116121372,0,697227,0,0,0,11115806,477382259,0,0,0,0,0,19266922,18177109,6373352,23392349,0,0,8718750,38622232,0,635761,115186475,11018173,112548197,0,1183364,0,0,0,0,6309689,"163,295,784.000",0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,106171049,ADVENTIST HEALTH CLEARLAKE,20211,2021-01-01,2021-03-31,Open,Lake,01 - Northern California,115,Non Profit Corp.,Comparable,Rural,707-995-5820,15630 18TH AVE,CLEARLAKE,95422,DAVID SANTOS,25,25,19,147,18,19,106,0,0,7,28,0,1,326,0,697,86,62,397,0,0,34,104,0,2,1382,0,36759,1946,8390,31076,0,0,10900,5139,0,1804,96014,11432241,1634602,513667,6690879,0,0,282398,16

All required columns are available.
Number of facilities: 461
Reporting periods: [np.int64(20211), np.int64(20212), np.int64(20213), np.int64(20214), np.int64(20221), np.int64(20222), np.int64(20223), np.int64(20224), np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]


## 4. Prepare the Future-Quarter Prediction Dataset

Each original row represents one hospital in one quarter. To create a forecasting dataset, we match:

- **Predictor information:** hospital data from the previous quarter.
- **Target:** `DIS_TOT` from the following quarter.

For example, 2025 Q4 hospital information is matched with 2026 Q1 `DIS_TOT`.

Only consecutive quarters are matched. This prevents a hospital's data from being incorrectly paired across a missing reporting period.

The hospital identifier (`FAC_NO`) is used for matching records, but it is not included as a model feature.

In [4]:
# Convert YEAR_QTR (for example, 20211) into a sortable quarter number.
# The resulting period number increases by one for each consecutive quarter.

def quarter_to_period(value):
    value = str(int(value))
    year = int(value[:4])
    quarter = int(value[-1])

    if quarter not in [1, 2, 3, 4]:
        raise ValueError(f"Unexpected quarter value: {value}")

    return year * 4 + (quarter - 1)


# Work with a copy so the original DataFrame remains unchanged.
df_model = df.copy()

# Remove rows without the information needed to identify the hospital or quarter.
df_model = df_model.dropna(subset=["FAC_NO", "YEAR_QTR"])

# Convert the reporting period into a numeric quarter sequence.
df_model["PERIOD_NUMBER"] = df_model["YEAR_QTR"].apply(quarter_to_period)

# Confirm that each hospital has no duplicate quarter records.
duplicate_count = df_model.duplicated(
    subset=["FAC_NO", "PERIOD_NUMBER"]
).sum()

print("Duplicate hospital-quarter records:", duplicate_count)

if duplicate_count > 0:
    raise ValueError("Duplicate hospital-quarter records must be investigated before matching quarters.")

Duplicate hospital-quarter records: 0


In [5]:
# Create a table containing the next quarter's target value.
target_next_quarter = df_model[
    ["FAC_NO", "PERIOD_NUMBER", "YEAR_QTR", "DIS_TOT"]
].copy()

target_next_quarter = target_next_quarter.rename(
    columns={
        "PERIOD_NUMBER": "TARGET_PERIOD_NUMBER",
        "YEAR_QTR": "TARGET_YEAR_QTR",
        "DIS_TOT": "DIS_TOT_NEXT_QTR"
    }
)

# For each previous-quarter record, identify the next consecutive quarter.
previous_quarter = df_model.copy()
previous_quarter["TARGET_PERIOD_NUMBER"] = (
    previous_quarter["PERIOD_NUMBER"] + 1
)

# Match each hospital's previous-quarter information to its next-quarter target.
model_df = previous_quarter.merge(
    target_next_quarter[
        ["FAC_NO", "TARGET_PERIOD_NUMBER", "TARGET_YEAR_QTR", "DIS_TOT_NEXT_QTR"]
    ],
    on=["FAC_NO", "TARGET_PERIOD_NUMBER"],
    how="inner"
)

# Keep rows with a known next-quarter target.
model_df = model_df.dropna(subset=["DIS_TOT_NEXT_QTR"]).copy()

# Rename the previous-quarter target so its timing is clear.
model_df = model_df.rename(
    columns={"DIS_TOT": "DIS_TOT_PREV_QTR"}
)

print("Future-quarter dataset shape:", model_df.shape)
print("Number of hospitals:", model_df["FAC_NO"].nunique())
print("Number of target quarters:", model_df["TARGET_YEAR_QTR"].nunique())

display(
    model_df[
        [
            "FAC_NO",
            "YEAR_QTR",
            "TARGET_YEAR_QTR",
            "DIS_TOT_PREV_QTR",
            "DIS_TOT_NEXT_QTR"
        ]
    ].head()
)

Future-quarter dataset shape: (8695, 163)
Number of hospitals: 460
Number of target quarters: 20


,FAC_NO,YEAR_QTR,TARGET_YEAR_QTR,DIS_TOT_PREV_QTR,DIS_TOT_NEXT_QTR
0,106580996,20211,20212,2868,3089
1,106150788,20211,20212,3721,3700
2,106171049,20211,20212,326,333
3,106150706,20211,20212,574,504
4,106190323,20211,20212,3893,4253


### Interpretation

The `YEAR_QTR` column identifies the quarter used for the predictor information. `TARGET_YEAR_QTR` identifies the following quarter being predicted.

For example, a row with `YEAR_QTR = 20254` and `TARGET_YEAR_QTR = 20261` uses 2025 Q4 information to predict 2026 Q1 discharges.

## 5. Select Candidate Features

The initial feature set is based on the EDA findings.

**Numerical features from the previous quarter:**
- `LIC_BEDS`
- `AVL_BEDS`
- `STF_BEDS`
- `DAY_TOT`
- `VIS_TOT`
- `DIS_TOT_PREV_QTR` (previous-quarter discharges)

**Categorical features from the previous quarter:**
- `TYPE_CNTRL`
- `TYPE_HOSP`
- `COUNTY_NAME`
- `HSA`

A bed availability ratio is also calculated from previous-quarter bed counts.

`TEACH_RURL` is not included in the initial feature set because the EDA found approximately 77.9% missing values. Discharge component variables are excluded because they are not needed for this forecasting design and may directly contribute to the discharge total.

The selected features are candidates for an initial model, not a final feature set.

In [6]:
# Create a capacity feature using previous-quarter bed counts.
model_df["BED_AVAILABILITY_RATIO"] = np.where(
    model_df["LIC_BEDS"] > 0,
    model_df["AVL_BEDS"] / model_df["LIC_BEDS"],
    np.nan
)

# Numerical predictors from the previous quarter.
numeric_features = [
    "LIC_BEDS",
    "AVL_BEDS",
    "STF_BEDS",
    "DAY_TOT",
    "VIS_TOT",
    "DIS_TOT_PREV_QTR",
    "BED_AVAILABILITY_RATIO"
]

# Categorical predictors from the previous quarter.
categorical_features = [
    "TYPE_CNTRL",
    "TYPE_HOSP",
    "COUNTY_NAME",
    "HSA"
]

# Keep only features that are present in the dataset.
numeric_features = [
    feature for feature in numeric_features
    if feature in model_df.columns
]

categorical_features = [
    feature for feature in categorical_features
    if feature in model_df.columns
]

feature_columns = numeric_features + categorical_features
target_column = "DIS_TOT_NEXT_QTR"

print("Numerical features:")
for feature in numeric_features:
    print("-", feature)

print("\nCategorical features:")
for feature in categorical_features:
    print("-", feature)

print("\nTarget:", target_column)

Numerical features:
- LIC_BEDS
- AVL_BEDS
- STF_BEDS
- DAY_TOT
- VIS_TOT
- DIS_TOT_PREV_QTR
- BED_AVAILABILITY_RATIO

Categorical features:
- TYPE_CNTRL
- TYPE_HOSP
- COUNTY_NAME
- HSA

Target: DIS_TOT_NEXT_QTR


## 6. Review Missing Values

Review missing values in the selected predictors and target.

Missing predictor values will be imputed later using statistics learned from the training data. Rows without a known target cannot be used for supervised model training and have already been excluded.

In [7]:
# Summarize missing values in the selected predictors.
missing_summary = pd.DataFrame({
    "Missing_Count": model_df[feature_columns].isna().sum(),
    "Missing_Percent": model_df[feature_columns].isna().mean() * 100
}).sort_values("Missing_Percent", ascending=False)

display(missing_summary)

print("Missing target values:", model_df[target_column].isna().sum())

,Missing_Count,Missing_Percent
BED_AVAILABILITY_RATIO,17,0.196
COUNTY_NAME,16,0.184
LIC_BEDS,0,0.000
AVL_BEDS,0,0.000
STF_BEDS,0,0.000
DAY_TOT,0,0.000
VIS_TOT,0,0.000
DIS_TOT_PREV_QTR,0,0.000
TYPE_CNTRL,0,0.000
TYPE_HOSP,0,0.000


Missing target values: 0


### Missing-value strategy

- Numerical predictors: replace missing values with the **training-set median**.
- Categorical predictors: replace missing values with the **training-set most frequent category**.
- Target (`DIS_TOT_NEXT_QTR`): do not impute; rows without a known target are excluded.

The imputation values will be learned from the training data only.

## 7. Split the Data Chronologically

Because this project predicts a future quarter, a random train/test split would not be appropriate. A chronological split better reflects how the model would be used in practice.

The target quarters are divided into:
- **Training set:** earliest 70% of target quarters.
- **Validation set:** following 15% of target quarters.
- **Testing set:** most recent 15% of target quarters.

All hospitals from the same target quarter remain in the same partition. The test set is kept separate for the final model evaluation.

In [8]:
# Sort the unique target quarters chronologically.
target_periods = sorted(
    model_df["TARGET_PERIOD_NUMBER"].dropna().unique()
)

n_periods = len(target_periods)

if n_periods < 6:
    raise ValueError(
        "There are too few target quarters for a reliable chronological "
        "training, validation, and testing split."
    )

# Allocate the earliest periods to training, followed by validation and testing.
train_end = int(n_periods * 0.70)
validation_end = int(n_periods * 0.85)

# Ensure each partition has at least one target quarter.
train_end = max(1, train_end)
validation_end = max(train_end + 1, validation_end)
validation_end = min(validation_end, n_periods - 1)

train_periods = target_periods[:train_end]
validation_periods = target_periods[train_end:validation_end]
test_periods = target_periods[validation_end:]

# Split by target quarter, not by individual row.
train_df = model_df[
    model_df["TARGET_PERIOD_NUMBER"].isin(train_periods)
].copy()

validation_df = model_df[
    model_df["TARGET_PERIOD_NUMBER"].isin(validation_periods)
].copy()

test_df = model_df[
    model_df["TARGET_PERIOD_NUMBER"].isin(test_periods)
].copy()

print("Training target quarters:", len(train_periods))
print("Validation target quarters:", len(validation_periods))
print("Testing target quarters:", len(test_periods))

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows:", len(test_df))

print("\nTraining target periods:", sorted(train_df["TARGET_YEAR_QTR"].unique()))
print("Validation target periods:", sorted(validation_df["TARGET_YEAR_QTR"].unique()))
print("Testing target periods:", sorted(test_df["TARGET_YEAR_QTR"].unique()))

Training target quarters: 14
Validation target quarters: 3
Testing target quarters: 3

Training rows: 6087
Validation rows: 1302
Testing rows: 1306

Training target periods: [np.int64(20212), np.int64(20213), np.int64(20214), np.int64(20221), np.int64(20222), np.int64(20223), np.int64(20224), np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243)]
Validation target periods: [np.int64(20244), np.int64(20251), np.int64(20252)]
Testing target periods: [np.int64(20253), np.int64(20254), np.int64(20261)]


In [9]:
# Confirm that the chronological partitions do not overlap.
train_set = set(train_periods)
validation_set = set(validation_periods)
test_set = set(test_periods)

assert train_set.isdisjoint(validation_set)
assert train_set.isdisjoint(test_set)
assert validation_set.isdisjoint(test_set)

# Confirm that the latest training period comes before validation,
# and validation comes before testing.
assert max(train_periods) < min(validation_periods)
assert max(validation_periods) < min(test_periods)

print("Chronological split checks passed.")

Chronological split checks passed.


## 8. Separate Predictors and Target

Separate the selected predictor columns (`X`) from the next-quarter discharge target (`y`).

The target remains in its original units (number of discharges). Only predictor features are imputed, encoded, and scaled.

In [10]:
# Training data.
X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

# Validation data.
X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target_column].copy()

# Testing data.
X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_validation shape:", y_validation.shape)
print("y_test shape:", y_test.shape)

X_train shape: (6087, 11)
X_validation shape: (1302, 11)
X_test shape: (1306, 11)

y_train shape: (6087,)
y_validation shape: (1302,)
y_test shape: (1306,)


## 9. Build the Preprocessing Pipelines

Use a `ColumnTransformer` to apply the appropriate preprocessing to each feature type.

**Numerical pipeline:**
1. Impute missing values with the median.
2. Standardize features using `StandardScaler`.

**Categorical pipeline:**
1. Impute missing values with the most frequent category.
2. Convert categories into indicator (dummy) variables using `OneHotEncoder`.

The encoder is configured to ignore categories that may appear in validation or test data but were not present in the training data.

In [11]:
# Preprocessing steps for numerical features.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing steps for categorical features.
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

# Apply the appropriate pipeline to each feature type.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop",
    sparse_threshold=0
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 10. Fit on Training Data and Transform All Sets

Fit the preprocessing pipeline using **only** the training data.

Then use the fitted pipeline to transform the training, validation, and testing predictors. This prevents information from the validation and testing periods from influencing imputation, encoding, or scaling.

In [12]:
# Learn imputation values, category levels, and scaling parameters from training data.
X_train_processed_array = preprocessor.fit_transform(X_train)

# Apply the fitted preprocessing steps to validation and testing data.
X_validation_processed_array = preprocessor.transform(X_validation)
X_test_processed_array = preprocessor.transform(X_test)

# Get the names of the transformed features.
processed_feature_names = preprocessor.get_feature_names_out()

# Convert the transformed arrays into DataFrames for easier inspection.
X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=processed_feature_names,
    index=X_train.index
)

X_validation_processed = pd.DataFrame(
    X_validation_processed_array,
    columns=processed_feature_names,
    index=X_validation.index
)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=processed_feature_names,
    index=X_test.index
)

print("Preprocessing completed.")
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_validation_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Preprocessing completed.
Processed training shape: (6087, 88)
Processed validation shape: (1302, 88)
Processed testing shape: (1306, 88)


## 11. Verify the Preprocessed Data

Check that:
- The transformed datasets have the same feature columns.
- Missing values have been handled.
- Numerical features have been standardized using training-set statistics.
- The target values remain separate from the predictors.

In [13]:
# Display a sample of the transformed training data.
display(X_train_processed.head())

# Check missing values after preprocessing.
print("Missing values in training predictors:", X_train_processed.isna().sum().sum())
print("Missing values in validation predictors:", X_validation_processed.isna().sum().sum())
print("Missing values in testing predictors:", X_test_processed.isna().sum().sum())

# Confirm that all partitions have the same transformed columns.
assert list(X_train_processed.columns) == list(X_validation_processed.columns)
assert list(X_train_processed.columns) == list(X_test_processed.columns)

# Check the means and standard deviations of the standardized numeric features.
scaled_numeric_columns = [
    column for column in X_train_processed.columns
    if column.startswith("num__")
]

display(
    X_train_processed[scaled_numeric_columns]
    .agg(["mean", "std"])
    .T
)

print("Preprocessing verification completed.")

,num__LIC_BEDS,num__AVL_BEDS,num__STF_BEDS,num__DAY_TOT,num__VIS_TOT,num__DIS_TOT_PREV_QTR,num__BED_AVAILABILITY_RATIO,cat__TYPE_CNTRL_City/County,cat__TYPE_CNTRL_District,cat__TYPE_CNTRL_Investor - Corp.,cat__TYPE_CNTRL_Investor - Indiv.,cat__TYPE_CNTRL_Investor - Ptnr.,cat__TYPE_CNTRL_Non Profit Corp.,cat__TYPE_CNTRL_Non Profit Other,cat__TYPE_CNTRL_State,cat__TYPE_HOSP_Hospital-LTC Emphasis,cat__TYPE_HOSP_Kaiser Foundation Health,cat__TYPE_HOSP_Psychiatric Health Facilities,cat__TYPE_HOSP_Shriners Hospitals,cat__TYPE_HOSP_State Hospitals,cat__COUNTY_NAME_Amador,cat__COUNTY_NAME_Butte,cat__COUNTY_NAME_Calaveras,cat__COUNTY_NAME_Colusa,cat__COUNTY_NAME_Contra Costa,cat__COUNTY_NAME_Del Norte,cat__COUNTY_NAME_El Dorado,cat__COUNTY_NAME_Fresno,cat__COUNTY_NAME_Glenn,cat__COUNTY_NAME_Humboldt,cat__COUNTY_NAME_Imperial,cat__COUNTY_NAME_Inyo,cat__COUNTY_NAME_Kern,cat__COUNTY_NAME_Kings,cat__COUNTY_NAME_Lake,cat__COUNTY_NAME_Lassen,cat__COUNTY_NAME_Los Angeles,cat__COUNTY_NAME_Madera,cat__COUNTY_NAME_Marin,cat__COUNTY_NAME_Mariposa,cat__COUNTY_NAME_Mendocino,cat__COUNTY_NAME_Merced,cat__COUNTY_NAME_Modoc,cat__COUNTY_NAME_Mono,cat__COUNTY_NAME_Monterey,cat__COUNTY_NAME_Napa,cat__COUNTY_NAME_Nevada,cat__COUNTY_NAME_Orange,cat__COUNTY_NAME_Placer,cat__COUNTY_NAME_Plumas,cat__COUNTY_NAME_Riverside,cat__COUNTY_NAME_Sacramento,cat__COUNTY_NAME_San Benito,cat__COUNTY_NAME_San Bernardino,cat__COUNTY_NAME_San Diego,cat__COUNTY_NAME_San Francisco,cat__COUNTY_NAME_San Joaquin,cat__COUNTY_NAME_San Luis Obispo,cat__COUNTY_NAME_San Mateo,cat__COUNTY_NAME_Santa Barbara,cat__COUNTY_NAME_Santa Clara,cat__COUNTY_NAME_Santa Cruz,cat__COUNTY_NAME_Shasta,cat__COUNTY_NAME_Siskiyou,cat__COUNTY_NAME_Solano,cat__COUNTY_NAME_Sonoma,cat__COUNTY_NAME_Stanislaus,cat__COUNTY_NAME_Sutter,cat__COUNTY_NAME_Tehama,cat__COUNTY_NAME_Trinity,cat__COUNTY_NAME_Tulare,cat__COUNTY_NAME_Tuolumne,cat__COUNTY_NAME_Ventura,cat__COUNTY_NAME_Yolo,cat__COUNTY_NAME_Yuba,cat__HSA_02 - Golden Empire,cat__HSA_03 - North Bay,cat__HSA_04 - West Bay,cat__HSA_05 - East Bay,cat__HSA_06 - North San Joaquin,cat__HSA_07 - Santa Clara,cat__HSA_08 - Mid-Coast,cat__HSA_09 - Central,cat__HSA_10 - Santa Barbara/Ventura,cat__HSA_11 - Los Angeles,cat__HSA_12 - Inland Counties,cat__HSA_13 - Orange,cat__HSA_14 - San Diego/Imperial
0,0.005,0.078,0.125,0.251,0.200,0.480,0.412,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.154,0.242,0.192,0.324,0.334,0.896,0.412,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000
2,-0.878,-0.895,-0.795,-0.738,1.090,-0.758,0.412,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,-0.287,-0.245,-0.462,-0.379,-0.072,-0.637,0.412,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.

Missing values in training predictors: 0
Missing values in validation predictors: 0
Missing values in testing predictors: 0


,mean,std
num__LIC_BEDS,-0.000,1.000
num__AVL_BEDS,-0.000,1.000
num__STF_BEDS,-0.000,1.000
num__DAY_TOT,0.000,1.000
num__VIS_TOT,0.000,1.000
num__DIS_TOT_PREV_QTR,-0.000,1.000
num__BED_AVAILABILITY_RATIO,0.000,1.000


Preprocessing verification completed.


## 12. Save the Prepared Datasets and Preprocessing Pipeline

Save the transformed predictors, target values, and fitted preprocessing pipeline in the processed-data folder.

These files can be loaded during model development without fitting the preprocessing steps again.

The test set remains untouched for final model evaluation.

In [14]:
# Folder for the prepared modeling datasets and preprocessing artifact.
OUTPUT_PATH = Path("../data/processed/modeling")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Save transformed predictors.
X_train_processed.to_csv(
    OUTPUT_PATH / "X_train_preprocessed.csv",
    index=False
)

X_validation_processed.to_csv(
    OUTPUT_PATH / "X_validation_preprocessed.csv",
    index=False
)

X_test_processed.to_csv(
    OUTPUT_PATH / "X_test_preprocessed.csv",
    index=False
)

# Save target values.
y_train.to_frame(name=target_column).to_csv(
    OUTPUT_PATH / "y_train.csv",
    index=False
)

y_validation.to_frame(name=target_column).to_csv(
    OUTPUT_PATH / "y_validation.csv",
    index=False
)

y_test.to_frame(name=target_column).to_csv(
    OUTPUT_PATH / "y_test.csv",
    index=False
)

# Save the fitted preprocessing pipeline.
joblib.dump(
    preprocessor,
    OUTPUT_PATH / "preprocessor.joblib"
)

print("Prepared datasets and preprocessing pipeline saved to:")
print(OUTPUT_PATH.resolve())

Prepared datasets and preprocessing pipeline saved to:
/Users/peiyeewhetsel/Documents/GitHub/DataScienceTrack/Capstones/PredictiveHealthcareOperations/Capstone2/data/processed/modeling


## 13. Summary

This notebook prepared the HCAI hospital data for a future-quarter discharge forecasting model.

The workflow:
- Matched each hospital's previous-quarter information with its next-quarter `DIS_TOT`.
- Excluded records without a consecutive-quarter match or a known target.
- Selected initial numerical and categorical predictors based on EDA findings.
- Created a bed availability ratio.
- Split target quarters chronologically into training, validation, and testing datasets.
- Imputed missing predictor values.
- Encoded categorical variables into indicator variables.
- Standardized numerical predictors.
- Saved the processed datasets and fitted preprocessing pipeline.

**Important modeling note:** The final test set should only be used for final model evaluation. Feature selection and model tuning should use the training and validation sets.

**Next step:** Develop and compare predictive models using the prepared training and validation datasets. Evaluate the selected model on the test set only after model selection is complete.